In [2]:
import os
import subprocess
import math
from enum import Enum
import random

import networkx as nx
import pyvis
import numpy as np
import scipy.stats as stats

import dash
from dash import dcc, html, Input, Output, State, callback_context as ctx
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import dashvis as dvis
import dashvis.stylesheets

In [30]:
%%script echo skipping 

# Build cpp-generate-graph
subprocess.run(["make", "cpp-generate-graph"], check=True, text=True, capture_output=True)

skipping


In [3]:
def load_degseqs() -> dict[dict]:
    categories = [
        "Metabolic",
        "Tissue",
        "Protein",
        "Connectome",
        "Food",
        "Rollcall",
        "Social",
        "Route",
        "CAIDA",
        "Circuit",
        "Technological",
        "Informational",
        "Transportation",
        "Economic",
        "Other",
    ]

    degseqs = {}
    for category in categories:
        degseqs[category] = {}

    for filename in os.listdir("SFAnalysis/degreesequences"):
        try:
            with open(os.path.join("SFAnalysis/degreesequences", filename), "r") as f:
                hist = {}
                for line in f.readlines()[1:]:
                    degree, count = map(int, line.strip().split(","))
                    hist[degree] = count

                graph_nodes_num = sum(hist.values())
                graph_edges_num = (
                    sum(degree * count for degree, count in hist.items()) // 2
                )
                graph_avg_deg = (
                    graph_edges_num / graph_nodes_num if graph_nodes_num > 0 else 0
                )
                graph_param_m = math.ceil(graph_avg_deg)

                category = "Other"
                for cat in categories:
                    if cat.lower() in filename.lower():
                        category = cat
                        break

                degseqs[category][filename] = {
                    "hist": hist,
                    "nodes": graph_nodes_num,
                    "edges": graph_edges_num,
                    "avg_deg": graph_avg_deg,
                    "param_m": graph_param_m,
                }

                found = False
                for category in categories:
                    if category in filename:
                        found = True
                        break
                if not found:
                    print(f"Category not found for file {filename}")
        except Exception as e:
            print(f"Error reading file {filename}: {e}")
            continue

    return degseqs


x = load_degseqs()
for category, graphs in x.items():
    print(f"{category}: {len(graphs)} graphs loaded")

Metabolic: 112 graphs loaded
Tissue: 526 graphs loaded
Protein: 39 graphs loaded
Connectome: 92 graphs loaded
Food: 117 graphs loaded
Rollcall: 413 graphs loaded
Social: 213 graphs loaded
Route: 733 graphs loaded
CAIDA: 1098 graphs loaded
Circuit: 60 graphs loaded
Technological: 62 graphs loaded
Informational: 26 graphs loaded
Transportation: 171 graphs loaded
Economic: 7 graphs loaded
Other: 0 graphs loaded


In [4]:
class NodeDistribution(Enum):
    STANDARD = 0
    CONSTANT = 1
    UNIFORM = 2
    PROPORTIONAL = 3
    POWER = 4
    NORMAL = 5
    LOG_NORMAL = 6
    EXPONENTIAL = 7
    LOG = 8
    WEIBULL = 9
    GAMMA = 10
    CHI_SQUARED = 11


class EdgeDistribution(Enum):
    STANDARD = 0
    CONSTANT = 1
    UNIFORM = 2
    ONE_OVER_DEGREE = 3
    ONE_OVER_DEGREE_LOG = 4


def generate_er_graph(n: int, p: float, filename: str) -> int:
    try:
        result = subprocess.run(
            ["./cpp-generate-graph", str(1), filename, str(n), str(p)],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        return -1

    return result.returncode


def generate_ba_graph(n0: int, n: int, m: int, filename: str) -> int:
    try:
        result = subprocess.run(
            ["./cpp-generate-graph", str(2), filename, str(n0), str(n), str(m)],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        return -1

    return result.returncode


def generate_ba_variate_graph(
    n0: int,
    n: int,
    m: int,
    filename: str,
    node_dist: int,
    edge_dist: int,
    args=[],
) -> int:
    try:
        result = subprocess.run(
            [
                "./cpp-generate-graph",
                str(3),
                filename,
                str(n0),
                str(n),
                str(m),
                str(node_dist),
                str(edge_dist),
            ]
            + [str(arg) for arg in args],
            check=True,
        )

    except subprocess.CalledProcessError as e:
        return -1

    return result.returncode


def read_graph_nx(filename: str) -> nx.Graph:
    try:
        with open(filename, "r") as f:
            graph_data = f.read()
    except Exception as e:
        print(f"Error reading file {filename}: {e}")
        return None

    graph = nx.Graph()
    for line in graph_data.splitlines():
        node1, node2 = [int(node) for node in line.strip().split()]
        graph.add_edge(node1, node2)

    return graph


def read_graph_pyvis(filename: str) -> pyvis.network.Network:
    graph_nx = read_graph_nx(filename)
    if graph_nx is None:
        return None

    graph = pyvis.network.Network()
    graph.from_nx(graph_nx)

    return graph


def read_graph_dashvis(filename: str, node_size: int) -> dict:
    graph_nx = read_graph_nx(filename)
    graph_object = {"nodes": [], "edges": []}

    if graph_nx is None:
        return graph_object

    for node, _ in graph_nx.nodes(data=True):
        graph_object["nodes"].append(
            {
                "id": node,
                "label": node,
                "shape": "dot",
                "size": graph_nx.degree[node] * node_size,
            }
        )

    for node1, node2, _ in graph_nx.edges(data=True):
        graph_object["edges"].append({"from": node1, "to": node2, "width": 1})

    return graph_object


def generate_degree_histogram_plot(graph: nx.Graph, example_hist: dict) -> go.Figure:
    degrees = [degree for _, degree in graph.degree()]
    fig = go.Figure(data=go.Histogram(x=degrees, name="Degree Histogram"))

    if example_hist:
        example_degrees = []
        for key, value in example_hist.items():
            example_degrees.extend([key] * value)
            
        fig.add_trace(
            go.Histogram(
                x=example_degrees,
                name="Example Histogram",
                opacity=0.75,
            )
        )
    
    fig.update_layout(
        title="Degree Histogram",
        xaxis_title="Degree",
        yaxis_title="Count",
        showlegend=True,
        legend=dict(
            x=0.8,
            y=0.85,
            xanchor='center',
            yanchor='middle',
        )
    )

    return fig


def generate_degree_comparison_plot(
    graph: nx.Graph, x_axis_type: str, y_axis_type: str, fit_power_law: bool, example_hist: dict
) -> go.Figure:
    degrees = [degree for _, degree in graph.degree()]
    degree_counts = np.bincount(degrees)[1:]
    cdf = np.cumsum(degree_counts[::-1])[::-1]
    cdf = cdf / cdf[0]

    nonzero_indices = degree_counts > 0
    k = np.arange(1, len(degree_counts) + 1)[nonzero_indices]
    cdf = cdf[nonzero_indices]

    log_k = np.log(k)
    log_cdf = np.log(cdf)

    # if fit_power_law:
    #     slope, intercept, r_value, p_value, std_err = stats.linregress(log_k, log_cdf)
    #     lambda_estimate = slope - 1
    #     estimated_power_law_text = f"Estimated power law: k^{lambda_estimate:.2f}"
    # else:
    #     estimated_power_law_text = ""

    fig = go.Figure(go.Scatter(x=k, y=cdf, mode="markers", name="Empirical CDF"))

    if example_hist:
        example_degrees = []
        for key, value in example_hist.items():
            example_degrees.extend([key] * value)
        example_degree_counts = np.bincount(example_degrees)[1:]
        example_cdf = np.cumsum(example_degree_counts[::-1])[::-1]
        example_cdf = example_cdf / example_cdf[0]

        example_nonzero_indices = example_degree_counts > 0
        example_k = np.arange(1, len(example_degree_counts) + 1)[example_nonzero_indices]
        example_cdf = example_cdf[example_nonzero_indices]

        example_log_k = np.log(example_k)
        example_log_cdf = np.log(example_cdf)

        fig.add_trace(
            go.Scatter(
                x=example_k,
                y=example_cdf,
                mode="markers",
                name="Example CDF",
            )
        )

    # if fit_power_law:
    #     fitted_cdf = np.exp(intercept) * k**slope
    #     fig.add_trace(
    #         go.Scatter(
    #             x=k,
    #             y=fitted_cdf,
    #             mode="lines",
    #             name="Fitted Power Law",
    #             line=dict(dash="dash"),
    #         )
    #     )

    fig.update_layout(
        # title=f"Degree CDF<br>{estimated_power_law_text}",
        title=f"Degree CDF",
        xaxis_title="Degree",
        yaxis_title="CDF",
        showlegend=False,
    )
    
    fig.update_xaxes(type=x_axis_type)
    fig.update_yaxes(type=y_axis_type)

    return fig

In [10]:
%%script echo skipping 

def run_power_law_interface(port: int) -> None:
    app = dash.Dash(
        __name__,
        suppress_callback_exceptions=True,
        external_stylesheets=[
            dbc.themes.SKETCHY,
            # dbc.themes.LITERA,
            dvis.stylesheets.VIS_NETWORK_STYLESHEET,
        ],
    )

    graph_visualisation = html.Div(
        id="graph-visualisation-containter",
        children=[
            html.Div(
                children=[
                    html.Label(
                        "n0:",
                        style={"marginTop": "10px", "marginRight": "25px"},
                    ),
                    dbc.Input(
                        id="graph-n0",
                        type="number",
                        value=10,
                        style={
                            "width": "90%",
                            "display": "flex",
                            "flexDirection": "row",
                            "alignItems": "center",
                            "marginLeft": "5%",
                        },
                    ),
                    html.Label(
                        "n:",
                        style={"marginTop": "10px", "marginRight": "25px"},
                    ),
                    dbc.Input(
                        id="graph-n",
                        type="number",
                        value=250,
                        style={
                            "width": "90%",
                            "display": "flex",
                            "flexDirection": "row",
                            "alignItems": "center",
                            "marginLeft": "5%",
                        },
                    ),
                    html.Label(
                        "m:",
                        style={"marginTop": "10px", "marginRight": "25px"},
                    ),
                    dbc.Input(
                        id="graph-m",
                        type="number",
                        value=3,
                        style={
                            "width": "90%",
                            "display": "flex",
                            "flexDirection": "row",
                            "alignItems": "center",
                            "marginLeft": "5%",
                        },
                    ),
                    html.Label(
                        "node size:",
                        style={"marginTop": "10px", "marginRight": "25px"},
                    ),
                    html.Div(
                        dcc.Slider(
                            id="graph-node-size",
                            min=1,
                            max=50,
                            value=25,
                            marks=None,
                            step=1,
                            tooltip={"placement": "bottom", "always_visible": True},
                        ),
                        style={"width": "150%", "margin": "0px 20px 20px 20px"},
                    ),
                    dbc.Button(
                        "Generate BA Graph",
                        id="generate-ba-graph",
                        style={
                            "width": "90%",
                            "display": "flex",
                            "flexDirection": "row",
                            "alignItems": "center",
                            "marginLeft": "5%",
                            "marginBottom": "15px",
                        },
                    ),
                    html.Span(
                        id="incorrect-graph-input",
                        className="badge bg-danger",
                        children="Incorrect Input",
                        style={"display": "none"},
                    ),
                ],
                style={
                    "width": "15%",
                    "height": "500px",
                    "display": "flex",
                    "flexDirection": "column",
                    "alignItems": "center",
                    "justifyContent": "center",
                    "paddingLeft": "50px",
                },
            ),
            dvis.DashNetwork(
                id="graph-visualisation",
                style={"height": "500px", "width": "85%"},
                data={"nodes": [], "edges": []},
                options={
                    "autoResize": True,
                    "physics": {
                        "forceAtlas2Based": {"springLength": 100},
                        "minVelocity": 0.5,
                        "solver": "forceAtlas2Based",
                    },
                    # "configure": {
                    #     "enabled": True,
                    #     "showButton": True,
                    # },
                },
            ),
        ],
        style={
            "width": "100%",
            "padding": "0",
            "display": "flex",
            "flexDirection": "row",
            "alignItems": "center",
        },
    )

    graph_properties = html.Div(
        id="graph-properties",
        children=[
            dcc.Graph(
                id="degrees-histogram", style={"width": "50%", "marginBottom": "100px"}
            ),
            html.Div(
                children=[
                    dcc.Graph(id="degrees-comparison", style={"width": "100%"}),
                    html.Div(
                        children=[
                            html.Label(
                                "x axis scale:",
                                style={
                                    "width": "50%",
                                    "textAlign": "right",
                                    "paddingRight": "10px",
                                },
                            ),
                            dcc.Dropdown(
                                id="graph-visualisation-x-axis-scale",
                                style={"width": "50%"},
                                options=[
                                    {"label": "Linear", "value": "linear"},
                                    {"label": "Log", "value": "log"},
                                ],
                                value="log",
                                clearable=False,
                            ),
                        ],
                        style={
                            "width": "100%",
                            "display": "flex",
                            "flexDirection": "row",
                            "alignItems": "center",
                            "justifyContent": "center",
                        },
                    ),
                    html.Div(
                        children=[
                            html.Label(
                                "y axis scale:",
                                style={
                                    "width": "50%",
                                    "textAlign": "right",
                                    "paddingRight": "10px",
                                },
                            ),
                            dcc.Dropdown(
                                id="graph-visualisation-y-axis-scale",
                                style={"width": "50%"},
                                options=[
                                    {"label": "Linear", "value": "linear"},
                                    {"label": "Log", "value": "log"},
                                ],
                                value="log",
                                clearable=False,
                            ),
                        ],
                        style={
                            "width": "100%",
                            "display": "flex",
                            "flexDirection": "row",
                            "alignItems": "center",
                            "justifyContent": "center",
                        },
                    ),
                ],
                style={
                    "width": "50%",
                    "padding": "20px",
                    "display": "flex",
                    "flexDirection": "column",
                    "alignItems": "center",
                    "justifyContent": "center",
                    "gap": "10px",
                },
            ),
        ],
        style={
            "width": "100%",
            "padding": "20px",
            "display": "flex",
            "flexDirection": "row",
            "alignItems": "center",
            "justifyContent": "center",
        },
    )

    app.layout = html.Div(
        [
            html.H1("Graphs Lab - Power Law", style={"padding": "20px"}),
            graph_visualisation,
            graph_properties,
        ],
        style={
            "minHeight": "95vh",
            "display": "flex",
            "flexDirection": "column",
            "alignItems": "center",
        },
    )

    @app.callback(
        Output("graph-visualisation", "data"),
        Output("degrees-histogram", "figure"),
        Output("degrees-comparison", "figure"),
        Output("incorrect-graph-input", "style"),
        Input("generate-ba-graph", "n_clicks"),
        State("graph-n0", "value"),
        State("graph-n", "value"),
        State("graph-m", "value"),
        State("graph-node-size", "value"),
        State("graph-visualisation-x-axis-scale", "value"),
        State("graph-visualisation-y-axis-scale", "value"),
        prevent_initial_call=True,
    )
    def _generate_ba_graph(n_clicks, n0, n, m, node_size, x_axis_scale, y_axis_scale):
        if n_clicks is None:
            return (dash.no_update,) * 4

        if generate_ba_graph(n0, n, m, "ba_graph.txt") != 0:
            return (dash.no_update,) * 3 + ({"display": "block"},)

        nx_graph = read_graph_nx("ba_graph.txt")
        dashvis_graph = read_graph_dashvis("ba_graph.txt", node_size / 10)
        hist = generate_degree_histogram_plot(nx_graph, {})
        comparison = generate_degree_comparison_plot(
            nx_graph, x_axis_scale, y_axis_scale, True, {}
        )

        return dashvis_graph, hist, comparison, {"display": "none"}

    app.run(
        port=port,
        jupyter_height=1300,
        jupyter_width="100%",
        debug=False,
        dev_tools_ui=False,
        dev_tools_props_check=False,
    )


run_power_law_interface(1117)

skipping


In [12]:
def run_power_law_variations_interface(port: int) -> None:
    app = dash.Dash(
        __name__,
        suppress_callback_exceptions=True,
        external_stylesheets=[
            dbc.themes.SKETCHY,
            # dbc.themes.LITERA,
            dvis.stylesheets.VIS_NETWORK_STYLESHEET,
        ],
    )

    node_dist_labels = {
        NodeDistribution.STANDARD.value: "selects all edges",
        NodeDistribution.UNIFORM.value: "P(k) = 1/n",
        NodeDistribution.PROPORTIONAL.value: "P(k) = k/2mn",
        NodeDistribution.POWER.value: "P(k) = k^(-α)",
        NodeDistribution.NORMAL.value: "P(k) = Normal(μ, σ)",
        NodeDistribution.LOG_NORMAL.value: "P(k) = Log-Normal(μ, σ)",
        NodeDistribution.EXPONENTIAL.value: "P(k) = Exponential(λ)",
        NodeDistribution.LOG.value: "P(k) = Log(base, k)",
        NodeDistribution.WEIBULL.value: "P(k) = Weibull(shape, scale)",
        NodeDistribution.GAMMA.value: "P(k) = Gamma(shape, scale)",
        NodeDistribution.CHI_SQUARED.value: "P(k) = Chi-Squared(df)",
    }

    node_dist_params_children = {
        NodeDistribution.STANDARD.value: [
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        NodeDistribution.UNIFORM.value: [
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        NodeDistribution.PROPORTIONAL.value: [
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        NodeDistribution.POWER.value: [
            {"display": "block"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        NodeDistribution.NORMAL.value: [
            {"display": "none"},
            {"display": "block"},
            {"display": "block"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        NodeDistribution.LOG_NORMAL.value: [
            {"display": "none"},
            {"display": "block"},
            {"display": "block"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        NodeDistribution.EXPONENTIAL.value: [
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "block"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        NodeDistribution.LOG.value: [
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "block"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        NodeDistribution.WEIBULL.value: [
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "block"},
            {"display": "block"},
            {"display": "none"},
        ],
        NodeDistribution.GAMMA.value: [
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "block"},
            {"display": "block"},
            {"display": "none"},
        ],
        NodeDistribution.CHI_SQUARED.value: [
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "block"},
        ],
    }

    edge_prob_labels = {
        EdgeDistribution.STANDARD.value: "selects all edges",
        EdgeDistribution.CONSTANT.value: "P(k) = p",
        EdgeDistribution.ONE_OVER_DEGREE.value: "P(k) = a/k^b + c",
        EdgeDistribution.ONE_OVER_DEGREE_LOG.value: "P(k) = a/log(k)^b + c",
    }

    edge_prob_params_children = {
        EdgeDistribution.STANDARD.value: [
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        EdgeDistribution.CONSTANT.value: [
            {"display": "block"},
            {"display": "none"},
            {"display": "none"},
            {"display": "none"},
        ],
        EdgeDistribution.ONE_OVER_DEGREE.value: [
            {"display": "none"},
            {"display": "block"},
            {"display": "block"},
            {"display": "block"},
        ],
        EdgeDistribution.ONE_OVER_DEGREE_LOG.value: [
            {"display": "none"},
            {"display": "block"},
            {"display": "block"},
            {"display": "block"},
        ],
    }

    graph_parameters = html.Div(
        id="graph-parameters",
        children=[
            html.Div(
                children=[
                    html.Div(
                        children=[
                            dcc.Graph(
                                id="node-attachment-graph",
                                style={
                                    "width": "100%",
                                    "height": "500px",
                                    "padding": "0",
                                    "margin": "0",
                                },
                            ),
                            dcc.Dropdown(
                                id="graph-node-distribution",
                                options=[
                                    {
                                        "label": "Standard",
                                        "value": NodeDistribution.STANDARD.value,
                                    },
                                    {
                                        "label": "Uniform",
                                        "value": NodeDistribution.UNIFORM.value,
                                    },
                                    {
                                        "label": "Proportional",
                                        "value": NodeDistribution.PROPORTIONAL.value,
                                    },
                                    {
                                        "label": "Power",
                                        "value": NodeDistribution.POWER.value,
                                    },
                                    {
                                        "label": "Normal",
                                        "value": NodeDistribution.NORMAL.value,
                                    },
                                    {
                                        "label": "Log-Normal",
                                        "value": NodeDistribution.LOG_NORMAL.value,
                                    },
                                    {
                                        "label": "Exponential",
                                        "value": NodeDistribution.EXPONENTIAL.value,
                                    },
                                    {
                                        "label": "Log",
                                        "value": NodeDistribution.LOG.value,
                                    },
                                    {
                                        "label": "Weibull",
                                        "value": NodeDistribution.WEIBULL.value,
                                    },
                                    {
                                        "label": "Gamma",
                                        "value": NodeDistribution.GAMMA.value,
                                    },
                                    {
                                        "label": "Chi-Squared",
                                        "value": NodeDistribution.CHI_SQUARED.value,
                                    },
                                ],
                                value=NodeDistribution.STANDARD.value,
                                clearable=False,
                                style={"width": "100%", "margin": "5px 0px"},
                            ),
                            html.Label(
                                id="graph-node-distribution-parameters-label",
                                className="text-secondary",
                                style={"marginLeft": "5px", "marginBottom": "20px"},
                            ),
                            html.Div(
                                id="graph-node-distribution-parameters-container",
                                children=[
                                    html.Div(
                                        id="graph-node-dist-alpha-container",
                                        children=[
                                            html.Label(
                                                "α:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-node-dist-alpha",
                                                type="number",
                                                value=2.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-node-dist-mean-container",
                                        children=[
                                            html.Label(
                                                "μ:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-node-dist-mean",
                                                type="number",
                                                value=0.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-node-dist-stddev-container",
                                        children=[
                                            html.Label(
                                                "σ:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-node-dist-stddev",
                                                type="number",
                                                value=1.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-node-dist-lambda-container",
                                        children=[
                                            html.Label(
                                                "λ:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-node-dist-lambda",
                                                type="number",
                                                value=1.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-node-dist-base-container",
                                        children=[
                                            html.Label(
                                                "base:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-node-dist-base",
                                                type="number",
                                                value=1.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-node-dist-shape-container",
                                        children=[
                                            html.Label(
                                                "shape:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-node-dist-shape",
                                                type="number",
                                                value=1.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-node-dist-scale-container",
                                        children=[
                                            html.Label(
                                                "scale:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-node-dist-scale",
                                                type="number",
                                                value=1.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-node-dist-freedom-container",
                                        children=[
                                            html.Label(
                                                "df:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-node-dist-freedom",
                                                type="number",
                                                value=1.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                ],
                            ),
                        ],
                        style={"width": "50%", "padding": "0 5%", "textAlign": "left"},
                    ),
                    html.Div(
                        children=[
                            dcc.Graph(
                                id="edge-existance-graph",
                                style={
                                    "width": "100%",
                                    "height": "500px",
                                    "padding": "0",
                                    "margin": "0",
                                },
                            ),
                            dcc.Dropdown(
                                id="graph-edge-probability",
                                options=[
                                    {
                                        "label": "Standard",
                                        "value": EdgeDistribution.STANDARD.value,
                                    },
                                    {
                                        "label": "Constant",
                                        "value": EdgeDistribution.CONSTANT.value,
                                    },
                                    {
                                        "label": "1 / Degree",
                                        "value": EdgeDistribution.ONE_OVER_DEGREE.value,
                                    },
                                    {
                                        "label": "1 / Log(Degree)",
                                        "value": EdgeDistribution.ONE_OVER_DEGREE_LOG.value,
                                    },
                                ],
                                value=EdgeDistribution.STANDARD.value,
                                clearable=False,
                                style={"width": "100%", "margin": "5px 0px"},
                            ),
                            html.Label(
                                id="graph-edge-probability-parameters-label",
                                className="text-secondary",
                                style={"marginLeft": "5px", "marginBottom": "20px"},
                            ),
                            html.Div(
                                id="graph-edge-probability-parameters-container",
                                children=[
                                    html.Div(
                                        id="graph-edge-dist-p-container",
                                        children=[
                                            html.Label(
                                                "p:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-edge-dist-p",
                                                type="number",
                                                value=10,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-edge-dist-a-container",
                                        children=[
                                            html.Label(
                                                "a:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-edge-dist-a",
                                                type="number",
                                                value=1.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-edge-dist-b-container",
                                        children=[
                                            html.Label(
                                                "b:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-edge-dist-b",
                                                type="number",
                                                value=1.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                    html.Div(
                                        id="graph-edge-dist-c-container",
                                        children=[
                                            html.Label(
                                                "c:",
                                                style={
                                                    "marginTop": "10px",
                                                    "marginRight": "25px",
                                                },
                                            ),
                                            dbc.Input(
                                                id="graph-edge-dist-c",
                                                type="number",
                                                value=1.0,
                                            ),
                                        ],
                                        style={
                                            "width": "90%",
                                            "display": "flex",
                                            "flexDirection": "row",
                                            "alignItems": "center",
                                            "marginLeft": "5%",
                                        },
                                    ),
                                ],
                            ),
                        ],
                        style={"width": "50%", "padding": "0 5%", "textAlign": "left"},
                    ),
                ],
                style={
                    "width": "100%",
                    "display": "flex",
                    "flexDirection": "row",
                    "marginBottom": "75px",
                },
            ),
            html.Div(
                children=[
                    html.Div(
                        children=[
                            html.Label(
                                "n0:",
                                style={
                                    "width": "40%",
                                    "textAlign": "left",
                                    "marginLeft": "30px",
                                },
                            ),
                            dbc.Input(
                                id="graph-n0",
                                type="number",
                                value=10,
                                style={"width": "60%", "margin": "0px 20px 20px 20px"},
                            ),
                        ],
                        style={"width": "25%"},
                    ),
                    html.Div(
                        children=[
                            html.Label(
                                "n:",
                                style={
                                    "width": "40%",
                                    "textAlign": "left",
                                    "marginLeft": "30px",
                                },
                            ),
                            dbc.Input(
                                id="graph-n",
                                type="number",
                                value=250,
                                style={"width": "60%", "margin": "0px 20px 20px 20px"},
                            ),
                        ],
                        style={"width": "25%"},
                    ),
                    html.Div(
                        children=[
                            html.Label(
                                "m:",
                                style={
                                    "width": "40%",
                                    "textAlign": "left",
                                    "marginLeft": "30px",
                                },
                            ),
                            dbc.Input(
                                id="graph-m",
                                type="number",
                                value=3,
                                style={"width": "60%", "margin": "0px 20px 20px 20px"},
                            ),
                        ],
                        style={"width": "25%"},
                    ),
                    html.Div(
                        children=[
                            html.Label(
                                "node size:",
                                style={
                                    "width": "40%",
                                    "textAlign": "left",
                                    "marginLeft": "30px",
                                },
                            ),
                            html.Div(
                                dcc.Slider(
                                    id="graph-node-size",
                                    min=1,
                                    max=50,
                                    value=25,
                                    marks=None,
                                    step=1,
                                    tooltip={
                                        "placement": "bottom",
                                        "always_visible": True,
                                    },
                                ),
                                style={"width": "90%", "margin": "15px 20px 20px 0px"},
                            ),
                        ],
                        style={"width": "25%"},
                    ),
                ],
                style={
                    "width": "100%",
                    "padding": "0",
                    "display": "flex",
                    "flexDirection": "row",
                },
            ),
        ],
        style={
            "width": "100%",
            "padding": "0",
            "display": "flex",
            "flexDirection": "column",
            "alignItems": "center",
        },
    )

    graph_visualisation = html.Div(
        id="graph-visualisation",
        children=[
            html.Div(
                children=[
                    dbc.Button(
                        "Generate Graph",
                        id="generate-graph",
                        style={"width": "100%", "marginBottom": "50px"},
                    ),
                    dcc.Dropdown(
                        id="examples-category-dropdown",
                        style={"width": "100%", "marginBottom": "5px"},
                        options=[
                            {"label": filename, "value": filename}
                            for filename in list(load_degseqs().keys())
                        ],
                        clearable=False,
                    ),
                    dcc.Dropdown(
                        id="examples-dropdown",
                        style={"width": "100%", "marginBottom": "5px"},
                        options=[],
                        clearable=False,
                    ),
                    dbc.Button(
                        "Pick Random Example",
                        id="pick-random-example",
                        style={"width": "100%", "marginBottom": "5px"},
                    ),
                    dbc.Button(
                        "Load Example",
                        id="load-example",
                        style={"width": "100%", "marginBottom": "10px"},
                    ),
                    dbc.Button(
                        "Load Example Without Graph",
                        id="load-example-without-graph",
                        style={"width": "100%", "marginBottom": "10px"},
                    ),
                    html.Span(
                        id="incorrect-graph-input",
                        className="badge bg-danger",
                        children="Incorrect Input",
                        style={"display": "none"},
                    ),
                ],
                style={
                    "width": "40%",
                    "display": "flex",
                    "flexDirection": "column",
                    "alignItems": "center",
                    "marginLeft": "5%",
                    "verticalAlign": "top",
                },
            ),
            dvis.DashNetwork(
                id="graph",
                style={"height": "500px", "width": "85%"},
                data={"nodes": [], "edges": []},
                options={
                    "autoResize": True,
                    "physics": {
                        "forceAtlas2Based": {"springLength": 100},
                        "minVelocity": 0.5,
                        "solver": "forceAtlas2Based",
                    },
                    # "configure": {
                    #     "enabled": True,
                    #     "showButton": True,
                    # },
                },
            ),
        ],
        style={
            "width": "100%",
            "height": "500px",
            "padding": "0",
            "display": "flex",
            "flexDirection": "row",
            "alignItems": "center",
        },
    )

    graph_properties = html.Div(
        id="graph-properties",
        children=[
            dcc.Graph(
                id="degrees-histogram",
                style={
                    "width": "50%",
                    "padding": "0",
                    "margin": "0",
                    "marginBottom": "100px",
                },
            ),
            html.Div(
                children=[
                    dcc.Graph(
                        id="degrees-comparison",
                        style={"width": "100%", "padding": "0", "margin": "0"},
                    ),
                    html.Div(
                        children=[
                            html.Label(
                                "x axis scale:",
                                style={
                                    "width": "50%",
                                    "textAlign": "right",
                                    "paddingRight": "10px",
                                },
                            ),
                            dcc.Dropdown(
                                id="graph-visualisation-x-axis-scale",
                                style={"width": "50%"},
                                options=[
                                    {"label": "Linear", "value": "linear"},
                                    {"label": "Log", "value": "log"},
                                ],
                                value="log",
                                clearable=False,
                            ),
                        ],
                        style={
                            "width": "100%",
                            "display": "flex",
                            "flexDirection": "row",
                            "alignItems": "center",
                            "justifyContent": "center",
                        },
                    ),
                    html.Div(
                        children=[
                            html.Label(
                                "y axis scale:",
                                style={
                                    "width": "50%",
                                    "textAlign": "right",
                                    "paddingRight": "10px",
                                },
                            ),
                            dcc.Dropdown(
                                id="graph-visualisation-y-axis-scale",
                                style={"width": "50%"},
                                options=[
                                    {"label": "Linear", "value": "linear"},
                                    {"label": "Log", "value": "log"},
                                ],
                                value="log",
                                clearable=False,
                            ),
                        ],
                        style={
                            "width": "100%",
                            "display": "flex",
                            "flexDirection": "row",
                            "alignItems": "center",
                            "justifyContent": "center",
                        },
                    ),
                ],
                style={
                    "width": "50%",
                    "padding": "20px",
                    "display": "flex",
                    "flexDirection": "column",
                    "alignItems": "center",
                    "justifyContent": "center",
                    "gap": "10px",
                },
            ),
        ],
        style={
            "width": "100%",
            "padding": "20px",
            "display": "flex",
            "flexDirection": "row",
            "alignItems": "center",
            "justifyContent": "center",
        },
    )

    app.layout = html.Div(
        [
            html.H1("Graphs Lab - Power Law Variations", style={"padding": "20px"}),
            graph_parameters,
            graph_visualisation,
            graph_properties,
        ],
        style={
            "minHeight": "95vh",
            "display": "flex",
            "flexDirection": "column",
            "alignItems": "center",
        },
    )

    generate_ba_graph(10, 1000, 3, "ba_graph.txt")
    g = read_graph_nx("ba_graph.txt")
    degree_range = [d for _, d in g.degree()]
    degree_range = list(range(1, max(degree_range) + 1))
    degseqs = load_degseqs()

    def generate_node_distribution_graph(
        x: list[int], y: list[float], scale: bool
    ) -> go.Figure:
        if scale:
            s = sum(y)
            y = [ys / s for ys in y]

        fig = go.Figure(
            data=[
                go.Scatter(
                    x=x,
                    y=y,
                    mode="markers",
                )
            ]
        )
        fig.update_layout(
            title=f"Node Attachment Distribution",
            xaxis_title="Node Degree (k)",
            yaxis_title="Probability P(k)",
            plot_bgcolor="rgba(0, 0, 0, 0)",
            paper_bgcolor="rgba(0, 0, 0, 0)",
        )

        return fig

    def generate_edge_probability_graph(x: list[int], y: list[float]) -> go.Figure:
        fig = go.Figure(
            data=[
                go.Scatter(
                    x=x,
                    y=y,
                    mode="markers",
                )
            ]
        )
        fig.update_layout(
            title=f"Edge Existence Probability",
            xaxis_title="Target Node Degree (k)",
            yaxis_title="Probability P(k)",
            plot_bgcolor="rgba(0, 0, 0, 0)",
            paper_bgcolor="rgba(0, 0, 0, 0)",
        )

        return fig

    @app.callback(
        Output("graph-node-distribution-parameters-label", "children"),
        Output("graph-node-dist-alpha-container", "style"),
        Output("graph-node-dist-mean-container", "style"),
        Output("graph-node-dist-stddev-container", "style"),
        Output("graph-node-dist-lambda-container", "style"),
        Output("graph-node-dist-base-container", "style"),
        Output("graph-node-dist-shape-container", "style"),
        Output("graph-node-dist-scale-container", "style"),
        Output("graph-node-dist-freedom-container", "style"),
        Input("graph-node-distribution", "value"),
    )
    def update_graph_node_distribution_parameters(selected_distribution):
        return node_dist_labels.get(
            selected_distribution
        ), *node_dist_params_children.get(selected_distribution)

    @app.callback(
        Output("graph-edge-probability-parameters-label", "children"),
        Output("graph-edge-dist-p-container", "style"),
        Output("graph-edge-dist-a-container", "style"),
        Output("graph-edge-dist-b-container", "style"),
        Output("graph-edge-dist-c-container", "style"),
        Input("graph-edge-probability", "value"),
    )
    def update_graph_edge_probability_parameters(selected_distribution):
        return edge_prob_labels.get(
            selected_distribution
        ), *edge_prob_params_children.get(selected_distribution)
        # return edge_prob_params_children.get(
        #     selected_distribution, []
        # ), edge_prob_labels.get(selected_distribution, "")

    @app.callback(
        Output("node-attachment-graph", "figure", allow_duplicate=True),
        Input("graph-node-distribution", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_node_attachment_graph_no_args(distribution):
        if distribution == NodeDistribution.STANDARD.value:
            return generate_node_distribution_graph(
                degree_range, [1.0 for _ in degree_range], False
            )
        elif distribution == NodeDistribution.UNIFORM.value:
            return generate_node_distribution_graph(
                degree_range, [1.0 for _ in degree_range], True
            )
        elif distribution == NodeDistribution.PROPORTIONAL.value:
            return generate_node_distribution_graph(
                degree_range, [k for k in degree_range], True
            )

        return dash.no_update

    @app.callback(
        Output("node-attachment-graph", "figure", allow_duplicate=True),
        Input("graph-node-dist-alpha", "value"),
        Input("graph-node-distribution", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_node_attachment_graph_power(alpha, distribution):
        if alpha is None or distribution != NodeDistribution.POWER.value:
            return dash.no_update

        return generate_node_distribution_graph(
            degree_range, [k**alpha for k in degree_range], True
        )

    @app.callback(
        Output("node-attachment-graph", "figure", allow_duplicate=True),
        Input("graph-node-dist-mean", "value"),
        Input("graph-node-dist-stddev", "value"),
        Input("graph-node-distribution", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_node_attachment_graph_mean(mean, stddev, distribution):
        if mean is None or stddev is None:
            return dash.no_update

        if distribution == NodeDistribution.NORMAL.value:
            return generate_node_distribution_graph(
                degree_range,
                [stats.norm.pdf(i, loc=mean, scale=stddev) for i in degree_range],
                True,
            )
        elif distribution == NodeDistribution.LOG_NORMAL.value:
            return generate_node_distribution_graph(
                degree_range,
                [
                    stats.lognorm.pdf(i, s=stddev, loc=0, scale=np.exp(mean))
                    for i in degree_range
                ],
                True,
            )

        return dash.no_update

    @app.callback(
        Output("node-attachment-graph", "figure", allow_duplicate=True),
        Input("graph-node-dist-lambda", "value"),
        Input("graph-node-distribution", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_node_attachment_graph_exponential(lambda_, distribution):
        if lambda_ is None or distribution != NodeDistribution.EXPONENTIAL.value:
            return dash.no_update

        return generate_node_distribution_graph(
            degree_range,
            [stats.expon.pdf(i, scale=1 / lambda_) for i in degree_range],
            True,
        )

    @app.callback(
        Output("node-attachment-graph", "figure", allow_duplicate=True),
        Input("graph-node-dist-base", "value"),
        Input("graph-node-distribution", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_node_attachment_graph_log(base, distribution):
        if base is None or distribution != NodeDistribution.LOG.value:
            return dash.no_update

        return generate_node_distribution_graph(
            # degree_range, [np.log(k + 1) / np.log(base) for k in degree_range], True
            degree_range,
            [np.log(k + 1) for k in degree_range],
            True,
        )

    @app.callback(
        Output("node-attachment-graph", "figure", allow_duplicate=True),
        Input("graph-node-dist-shape", "value"),
        Input("graph-node-dist-scale", "value"),
        Input("graph-node-distribution", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_node_attachment_graph_weibull_gamma(shape, scale, distribution):
        if shape is None or scale is None:
            return dash.no_update

        if distribution == NodeDistribution.WEIBULL.value:
            return generate_node_distribution_graph(
                degree_range,
                [stats.weibull_min.pdf(i, c=shape, scale=scale) for i in degree_range],
                True,
            )
        elif distribution == NodeDistribution.GAMMA.value:
            return generate_node_distribution_graph(
                degree_range,
                [stats.gamma.pdf(i, a=shape, scale=scale) for i in degree_range],
                True,
            )

        return dash.no_update

    @app.callback(
        Output("node-attachment-graph", "figure", allow_duplicate=True),
        Input("graph-node-dist-freedom", "value"),
        Input("graph-node-distribution", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_node_attachment_graph_chi_squared(freedom, distribution):
        if freedom is None or distribution != NodeDistribution.CHI_SQUARED.value:
            return dash.no_update

        return generate_node_distribution_graph(
            degree_range, [stats.chi2.pdf(k, freedom) for k in degree_range], True
        )

    @app.callback(
        Output("edge-existance-graph", "figure", allow_duplicate=True),
        Input("graph-edge-probability", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_edge_existance_graph_no_args(distribution):
        if distribution == EdgeDistribution.STANDARD.value:
            return generate_edge_probability_graph(
                degree_range,
                [1.0 for _ in degree_range],
            )

        return dash.no_update

    @app.callback(
        Output("edge-existance-graph", "figure", allow_duplicate=True),
        Input("graph-edge-dist-p", "value"),
        Input("graph-edge-probability", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_edge_existance_graph_constant(p, distribution):
        if p is None or distribution != EdgeDistribution.CONSTANT.value:
            return dash.no_update

        return generate_edge_probability_graph(
            degree_range,
            [p for _ in degree_range],
        )

    @app.callback(
        Output("edge-existance-graph", "figure", allow_duplicate=True),
        Input("graph-edge-dist-a", "value"),
        Input("graph-edge-dist-b", "value"),
        Input("graph-edge-dist-c", "value"),
        State("graph-edge-probability", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_edge_existance_graph_one_over(a, b, c, distribution):
        if a is None or b is None or c is None:
            return dash.no_update

        if distribution == EdgeDistribution.ONE_OVER_DEGREE.value:
            return generate_edge_probability_graph(
                degree_range,
                [(a / k**b + c) for k in degree_range],
            )
        elif distribution == EdgeDistribution.ONE_OVER_DEGREE_LOG.value:
            return generate_edge_probability_graph(
                degree_range,
                [(a / (np.log(k) ** b) + c) for k in degree_range],
            )

        return dash.no_update

    @app.callback(
        Output("generate-graph", "n_clicks", allow_duplicate=True),
        Output("graph", "data"),
        Output("degrees-histogram", "figure"),
        Output("degrees-comparison", "figure"),
        Output("incorrect-graph-input", "style"),
        Input("generate-graph", "n_clicks"),
        State("graph-n0", "value"),
        State("graph-n", "value"),
        State("graph-m", "value"),
        State("graph-node-distribution", "value"),
        State("graph-edge-probability", "value"),
        State("graph-node-size", "value"),
        State("graph-visualisation-x-axis-scale", "value"),
        State("graph-visualisation-y-axis-scale", "value"),
        Input("graph-node-dist-lambda", "value"),
        State("graph-node-dist-alpha", "value"),
        State("graph-node-dist-mean", "value"),
        State("graph-node-dist-stddev", "value"),
        State("graph-node-dist-base", "value"),
        State("graph-node-dist-shape", "value"),
        State("graph-node-dist-scale", "value"),
        State("graph-node-dist-freedom", "value"),
        State("graph-edge-dist-p", "value"),
        State("graph-edge-dist-a", "value"),
        State("graph-edge-dist-b", "value"),
        State("graph-edge-dist-c", "value"),
        State("examples-category-dropdown", "value"),
        State("examples-dropdown", "value"),
        prevent_initial_call="initial_duplicate",
    )
    def update_graph_data(
        n_clicks,
        n0,
        n,
        m,
        node_distribution,
        edge_probability,
        node_size,
        x_axis_scale,
        y_axis_scale,
        param_lambda,
        param_alpha,
        param_mean,
        param_stddev,
        param_base,
        param_shape,
        param_scale,
        param_freedom,
        param_p,
        param_a,
        param_b,
        param_c,
        example_category,
        example
    ):
        if n_clicks is None:
            return (dash.no_update,) * 5
        
        args = []
        if node_distribution == NodeDistribution.STANDARD.value:
            pass
        elif node_distribution == NodeDistribution.UNIFORM.value:
            pass
        elif node_distribution == NodeDistribution.PROPORTIONAL.value:
            pass
        elif node_distribution == NodeDistribution.POWER.value:
            args.append(param_alpha)
        elif node_distribution == NodeDistribution.NORMAL.value:
            args.extend([param_mean, param_stddev])
        elif node_distribution == NodeDistribution.LOG_NORMAL.value:
            args.extend([param_mean, param_stddev])
        elif node_distribution == NodeDistribution.EXPONENTIAL.value:
            args.append(param_lambda)
        elif node_distribution == NodeDistribution.LOG.value:
            args.append(param_base)
        elif node_distribution == NodeDistribution.WEIBULL.value:
            args.extend([param_shape, param_scale])
        elif node_distribution == NodeDistribution.GAMMA.value:
            args.extend([param_shape, param_scale])
        elif node_distribution == NodeDistribution.CHI_SQUARED.value:
            args.append(param_freedom)

        if edge_probability == EdgeDistribution.STANDARD.value:
            pass
        elif edge_probability == EdgeDistribution.CONSTANT.value:
            args.append(param_p)
        elif edge_probability == EdgeDistribution.ONE_OVER_DEGREE.value:
            args.extend([param_a, param_b, param_c])
        elif edge_probability == EdgeDistribution.ONE_OVER_DEGREE_LOG.value:
            args.extend([param_a, param_b, param_c])

        if (
            generate_ba_variate_graph(
                n0, n, m, "ba_graph.txt", node_distribution, edge_probability, args
            )
            != 0
        ):
            return (dash.no_update,) * 4 + ({"display": "block"},)

        if n_clicks < 0:
            example_hist = load_degseqs().get(example_category, {}).get(example, None).get("hist", {})
        else:
            example_hist = None

        nx_graph = read_graph_nx("ba_graph.txt")
        dashvis_graph = read_graph_dashvis("ba_graph.txt", node_size / 10)
        hist = generate_degree_histogram_plot(nx_graph, example_hist)
        comparison = generate_degree_comparison_plot(
            nx_graph, x_axis_scale, y_axis_scale, True, example_hist
        )

        if n_clicks == -2:
            return 1, dash.no_update, hist, comparison, {"display": "none"}
        else:
            return 1, dashvis_graph, hist, comparison, {"display": "none"}

    @app.callback(
        Output("examples-dropdown", "options"),
        Input("examples-category-dropdown", "value"),
        prevent_initial_call=True,
    )
    def update_examples_dropdown(selected_category):
        if not selected_category:
            return dash.no_update

        examples = load_degseqs().get(selected_category, [])
        return [{"label": filename, "value": filename} for filename in examples]

    @app.callback(
        Output("examples-dropdown", "value"),
        Input("pick-random-example", "n_clicks"),
        State("examples-category-dropdown", "value"),
        prevent_initial_call=True,
    )
    def pick_random_example(n_clicks, selected_category):
        if n_clicks is None or not selected_category:
            return dash.no_update

        examples = load_degseqs().get(selected_category, [])
        if not examples:
            return dash.no_update

        example_key = list(examples.keys())

        return random.choice(example_key)

    @app.callback(
        Output("graph-n", "value"),
        Output("graph-m", "value"),
        Output("generate-graph", "n_clicks", allow_duplicate=True),
        Input("load-example", "n_clicks"),
        Input("load-example-without-graph", "n_clicks"),
        State("examples-category-dropdown", "value"),
        State("examples-dropdown", "value"),
        prevent_initial_call=True,
    )
    def load_example(n_clicks1, n_clicks2, selected_category, selected_example):
        if (
            n_clicks1 is None
            and n_clicks2 is None
            or not selected_category
            or not selected_example
        ):
            return dash.no_update, dash.no_update, dash.no_update

        examples = load_degseqs().get(selected_category, {})
        if not examples:
            return dash.no_update, dash.no_update, dash.no_update

        example_data = examples.get(selected_example, None)
        if not example_data:
            return dash.no_update, dash.no_update, dash.no_update

        nodes_num = example_data["nodes"]
        edges_num = example_data["edges"]
        avg_deg = example_data["avg_deg"]
        param_m = example_data["param_m"]

        if "load-example-without-graph" in ctx.triggered_id:
            return nodes_num, param_m, -2
        return nodes_num, param_m, -1

    app.run(
        port=port,
        jupyter_height=2300,
        jupyter_width="100%",
        debug=False,
        dev_tools_ui=False,
        dev_tools_props_check=False,
    )


run_power_law_variations_interface(1118)